# 🎨 AquaRender — Watercolor on Kaggle or Colab

Same notebook, both environments. Auto-detects which one you're on. Powered by
**FLUX.1-dev** + the SebastianBodza Aquarell watercolor LoRA — produces real painterly
watercolor output (loose washes, paper texture, soft pastel hues) instead of
the smooth digital illustration that schnell tended to generate.

**Three steps:**
1. Right pane → enable a **GPU** + **Internet** *(see below for each platform)*.
2. **Run All**. First run downloads a single 17 GB FLUX checkpoint (~7 min — 17 GB checkpoint + 171 MB LoRA). Re-runs are seconds.
3. Pick an image (URL or upload), pick a style, click **Generate**. Result inline + **📥 Download** button.

| Platform | GPU setting | Internet setting |
|----------|-------------|------------------|
| **Kaggle** | Right pane → Settings → Accelerator: **GPU P100** | Right pane → Settings → Internet: **On** |
| **Colab**  | Top menu → Runtime → **Change runtime type** → Hardware accelerator: **T4 GPU** | Always on by default |

GPU memory: FLUX schnell at fp8 needs ~12 GB during sampling — fits T4 (15 GB)
and P100 (16 GB) comfortably. ComfyUI auto-offloads the T5 text encoder to CPU.

> Repo: https://github.com/8lianno/aquarender · MIT.


In [ ]:
# ── 1/2 Engine setup — runs once per session, idempotent on re-run ──
import os, sys, subprocess, pathlib, time, json, urllib.request

# Detect which free-GPU host we're on. Falls back to local for dev.
if pathlib.Path("/kaggle/working").exists():
    ENV = "kaggle"
    BASE = pathlib.Path("/kaggle/working")
    USER_DATA = pathlib.Path("/kaggle/input")
elif pathlib.Path("/content").exists():
    ENV = "colab"
    BASE = pathlib.Path("/content")
    USER_DATA = None  # Colab: mount Drive separately if you want a LoRA dataset
else:
    ENV = "local"
    BASE = pathlib.Path.cwd() / "aquarender_runtime"
    BASE.mkdir(exist_ok=True)
    USER_DATA = None

print(f"✓ Environment: {ENV}  (base: {BASE})")

COMFY_DIR = BASE / "ComfyUI"
LOG_PATH = BASE / "comfyui.log"
OUTPUT_PATH = BASE / "aquarender_output.png"

# 1. GPU check (works the same on Kaggle and Colab)
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
    ).decode().strip()
    print(f"✓ GPU: {gpu}")
except Exception:
    if ENV == "kaggle":
        msg = "Settings → Accelerator → GPU P100, then Run All again."
    elif ENV == "colab":
        msg = "Runtime → Change runtime type → Hardware accelerator: T4 GPU, then Run All again."
    else:
        msg = "No NVIDIA GPU found locally."
    raise SystemExit(f"✗ No GPU. {msg}")

# 2. Clone ComfyUI (shallow) — idempotent. We track main rather than a
# specific commit; pinning by SHA via `git fetch --depth=1 origin <sha>`
# requires `uploadpack.allowReachableSHA1InWant=true` server-side and is
# brittle. Cross-session reproducibility was never guaranteed anyway
# (different GPU classes drift slightly) — see docs/ARCHITECTURE.md.
if not COMFY_DIR.exists():
    print("⏬ Cloning ComfyUI…")
    subprocess.check_call(
        ["git", "clone", "--quiet", "--depth=1",
         "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)]
    )
subprocess.check_call(["pip", "install", "--quiet", "-r", str(COMFY_DIR / "requirements.txt")])
print("✓ ComfyUI ready")

# 3. Download models with visible progress bars
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

MODELS = [
    # FLUX.1-dev (fp8, bundled UNet + T5 + CLIP + VAE in one safetensors).
    # Higher quality than schnell on painterly styles — schnell is a 4-step
    # distilled fast-mode model that produces clean digital illustration but
    # struggles with loose watercolor / pastel / oil texture. dev is the
    # full guidance model: 20 steps, ~50 s/image on T4, much more painterly.
    # License: BFL Non-Commercial License (free for personal use, not for
    # selling outputs commercially). Apache-2.0 schnell remains an option;
    # see git history if you want to swap back.
    ("https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
     "checkpoints/flux1-dev-fp8.safetensors"),
    # SebastianBodza/Flux_Aquarell_Watercolor_v2 — top-rated FLUX-dev
    # watercolor LoRA. 171 MB. Trigger word: "AQUACOLTOK". Used by every
    # preset; cell 3's LORAS registry auto-injects the trigger.
    ("https://huggingface.co/SebastianBodza/Flux_Aquarell_Watercolor_v2/resolve/main/lora.safetensors",
     "loras/flux_aquarell_v2.safetensors"),
    # alvdansen/softpasty-flux-dev — soft pastel illustration style. 451 MB.
    # Trigger: "araminta_illus illustration style". Stacks with the
    # aquarell LoRA on the "Pastel Watercolor" preset for a real
    # painted-medium feel (washes + paper texture + soft pastel hues).
    ("https://huggingface.co/alvdansen/softpasty-flux-dev/resolve/main/araminta_k_softpasty_diffusion_flux.safetensors",
     "loras/flux_softpasty.safetensors"),
]

def fetch(url, rel):
    dest = COMFY_DIR / "models" / rel
    if dest.exists():
        print(f"  ✓ {dest.name} ({dest.stat().st_size / 1e6:.0f} MB) cached")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as r:
        total = int(r.headers.get("Content-Length", 0))
        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, unit_divisor=1024,
            desc=dest.name[:40], leave=False,
        ) as pbar:
            while True:
                chunk = r.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))
    print(f"  ✓ {dest.name} downloaded")

print("⏬ Downloading models (skips cached)…")
for url, rel in MODELS:
    fetch(url, rel)

# 4. Symlink user-uploaded LoRA datasets (Kaggle convention)
if USER_DATA is not None and USER_DATA.exists():
    for ds in USER_DATA.glob("*"):
        if not ds.is_dir():
            continue
        for f in ds.rglob("*.safetensors"):
            link = COMFY_DIR / "models/loras" / f.name
            if not link.exists():
                try:
                    os.symlink(f, link)
                    print(f"  ↳ linked {f.name} from {USER_DATA}/{ds.name}/")
                except OSError:
                    pass

# 5. (Re)launch ComfyUI on 127.0.0.1:8188
try:
    subprocess.check_call(["pkill", "-f", "ComfyUI/main.py"], stderr=subprocess.DEVNULL)
    time.sleep(2)
except subprocess.CalledProcessError:
    pass
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd=str(COMFY_DIR),
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)
print(f"✓ ComfyUI launching (PID {proc.pid}, log: {LOG_PATH})")

# 6. Wait for /system_stats
print("⏳ Waiting for engine", end="", flush=True)
deadline = time.time() + 240
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            v = json.load(r).get("system", {}).get("comfyui_version", "?")
            print(f" → ✓ ComfyUI {v} ready")
            break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    raise SystemExit(f"✗ Engine never came up. tail {LOG_PATH}")

print("\n🎨 Setup complete — scroll down to generate.")


In [ ]:
# ── 2/2 Generate — pick input, click button, download result ──
import os, time, json, base64, urllib.parse, pathlib
import requests
import ipywidgets as widgets
from IPython.display import display, Image, HTML

# OUTPUT_PATH was set in cell 2 based on environment (Kaggle vs Colab vs local)
COMFY_URL = "http://127.0.0.1:8188"

# ── Tuning notes (learned the hard way) ──
# At denoise > 0.80 + a strong style LoRA + FluxGuidance 3.5, FLUX-dev
# img2img discards the source latent and generates a generic painting from
# the prompt instead. User uploaded a bearded selfie and got back a young
# woman with freckles — softpasty LoRA's signature character drift won over
# subject identity. Subject-preservation rules of thumb:
#   denoise 0.55–0.72  : source dominates, painterly style overlaid
#   denoise 0.72–0.85  : equal balance, identity may drift
#   denoise > 0.85     : prompt dominates, near-text-to-image
# When stacking LoRAs keep total weight ≤ 1.2; softpasty in particular
# pushes faces toward stylized young features and should stay ≤ 0.4.
# FluxGuidance: 2.5 for img2img (3.5 is txt2img-painterly).
#
# ── LoRA registry ──
# Single source of truth for every LoRA we ship: filename, trigger word,
# default weight. Adding a new LoRA = one entry here + one MODELS line in
# cell 2 + listing it in a preset's `loras=[...]`. _build_workflow chains
# LoraLoader nodes for each and auto-prepends the trigger to the prompt
# (skipped if the user already typed it manually).
LORAS = {
    "flux_aquarell_v2.safetensors": {
        "trigger": "AQUACOLTOK",
        "default_weight": 0.95,
        # README also recommends "white background" alongside; the prompt
        # text in each preset already has it.
    },
    "flux_softpasty.safetensors": {
        "trigger": "araminta_illus illustration style",
        "default_weight": 0.85,
    },
}

# Each preset declares which LoRA(s) to load. Form: a list of either
#   "filename.safetensors"            → use the registry's default_weight
#   ("filename.safetensors", 0.75)    → use this explicit weight
PRESETS = {
    "Soft Watercolor (portraits, lifestyle)": dict(
        prompt=("A loose, expressive watercolor painting on textured cold-press paper. "
                "Soft pastel washes, gestural brushstrokes, organic color bleeds, "
                "lots of white paper visible through the painting, painterly, hand-painted, "
                "traditional watercolor. White background."),
        loras=["flux_aquarell_v2.safetensors"],
        denoise=0.65,
    ),
    "Pastel Watercolor (painted medium feel — stacks both LoRAs)": dict(
        prompt=("A loose painterly portrait sketch in real watercolor and pastel media, "
                "soft pastel washes, organic color bleeds, visible cold-press paper texture, "
                "gestural confident strokes, hand-painted in a sketchbook, sketchy edges, "
                "lots of white paper showing through. White background."),
        # Stacking aquarell + softpasty: aquarell drives the watercolor wash,
        # softpasty adds the soft pastel illustration feel. Both at reduced
        # weights so they don't fight each other.
        loras=[
            ("flux_aquarell_v2.safetensors", 0.85),
            # softpasty stays at low weight: enough to add pastel feel,
            # not enough to override subject identity (it was trained on
            # stylized young female faces, dominates at higher weights).
            ("flux_softpasty.safetensors", 0.35),
        ],
        denoise=0.68,
    ),
    "Ink + Watercolor (architecture, urban)": dict(
        prompt=("An ink and watercolor painting, expressive black ink linework "
                "with soft watercolor color washes overlaid. Architectural sketchbook style, "
                "hand-drawn ink lines, visible paper texture, loose confident strokes. "
                "White background."),
        loras=["flux_aquarell_v2.safetensors"],
        denoise=0.60,
    ),
    "Children's Book (characters, animals)": dict(
        prompt=("A whimsical watercolor children's book illustration, warm "
                "pastel colors, loose painterly brushstrokes, charming hand-painted "
                "storybook art with soft edges. White background."),
        loras=["flux_aquarell_v2.safetensors"],
        denoise=0.72,
    ),
    "Product Watercolor (ecommerce)": dict(
        prompt=("A watercolor product illustration, soft pastel color wash, "
                "clean white paper background, hand-painted, gentle painterly brushstrokes, "
                "minimal detail. White background."),
        loras=["flux_aquarell_v2.safetensors"],
        denoise=0.55,
    ),
}
# Strength scales denoise. FLUX-dev with a strong watercolor LoRA already
# reinterprets the source heavily, so even Light gets you a real painting
# instead of a smooth digital illustration.
STRENGTH_MULT = {
    "Light":  0.85,
    "Medium": 1.00,
    "Strong": 1.15,
}


def _resolve_loras(loras):
    """Normalise a preset's `loras=[...]` entry into [(filename, weight, trigger)]
    by looking up each filename in the LORAS registry."""
    out = []
    for entry in loras:
        filename = entry[0] if isinstance(entry, tuple) else entry
        if filename not in LORAS:
            raise ValueError(
                f"LoRA {filename!r} used by a preset but missing from LORAS registry — "
                f"add it (with its trigger word) to the registry above."
            )
        weight = entry[1] if isinstance(entry, tuple) else LORAS[filename]["default_weight"]
        out.append((filename, weight, LORAS[filename]["trigger"]))
    return out


def _inject_triggers(prompt, triggers):
    """Auto-prepend each LoRA's trigger word(s) to the prompt, skipping any
    that are already present (so a hand-tuned prompt with the trigger doesn't
    get a duplicate)."""
    missing = [t for t in triggers if t.lower() not in prompt.lower()]
    if not missing:
        return prompt
    return ", ".join(missing) + ". " + prompt

# ── Widgets ──
input_mode = widgets.RadioButtons(
    options=["URL", "Upload"], value="URL", description="Input:",
    layout=widgets.Layout(width="auto"),
)
url_box = widgets.Text(
    value="https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/640px-PNG_transparency_demonstration_1.png",
    placeholder="https://…", description="Image URL:",
    layout=widgets.Layout(width="700px"),
)
file_uploader = widgets.FileUpload(
    accept="image/*", multiple=False, description="Upload image",
    layout=widgets.Layout(width="auto"),
)
preset_dd = widgets.Dropdown(
    options=list(PRESETS.keys()), value="Soft Watercolor (portraits, lifestyle)",
    description="Style:", layout=widgets.Layout(width="450px"),
)
strength_dd = widgets.Dropdown(
    options=["Light", "Medium", "Strong"], value="Medium", description="Strength:",
    layout=widgets.Layout(width="220px"),
)
seed_box = widgets.IntText(
    value=0, description="Seed (0=random):", layout=widgets.Layout(width="220px"),
)
btn = widgets.Button(
    description="🎨 Generate watercolor", button_style="primary",
    layout=widgets.Layout(width="300px", height="40px"),
)
progress_bar = widgets.IntProgress(
    min=0, max=100, value=0, bar_style="info",
    layout=widgets.Layout(width="700px", visibility="hidden"),
)
status_html = widgets.HTML("")
output_box = widgets.Output()


def _load_input_image() -> bytes:
    if input_mode.value == "URL":
        url = url_box.value.strip()
        if not url.startswith(("http://", "https://")):
            raise ValueError("URL must start with http:// or https://")
        return requests.get(url, timeout=30).content
    v = file_uploader.value
    if not v:
        raise ValueError("Click 'Upload image' first, or switch to URL mode")
    if isinstance(v, tuple):  # ipywidgets v8
        return bytes(v[0]["content"])
    if isinstance(v, dict):  # ipywidgets v7
        return bytes(next(iter(v.values()))["content"])
    raise ValueError(f"Unexpected upload widget value type: {type(v)}")


def _build_workflow(image_name, prompt, loras, denoise, seed):
    """FLUX.1-dev img2img with one or more LoRAs stacked.

    LoraLoader nodes are chained dynamically based on the preset's `loras`
    list — first one consumes Checkpoint(model, clip), each subsequent one
    consumes the previous LoraLoader's outputs. The final model+clip
    pointers feed into KSampler and CLIPTextEncode respectively.

    dev requires KSampler.cfg=1 + a separate FluxGuidance(2.5) node carrying
    the real guidance scale. 20 steps, sampler=euler, scheduler=simple is the
    canonical FLUX-dev recipe for painterly output.
    """
    resolved = _resolve_loras(loras)
    full_prompt = _inject_triggers(prompt, [t for _, _, t in resolved])

    wf = {
        "1": {"class_type":"CheckpointLoaderSimple",
              "inputs":{"ckpt_name":"flux1-dev-fp8.safetensors"}},
    }

    # Chain LoraLoader nodes (ids 100, 101, …). Each consumes the previous
    # model+clip refs; the last one's outputs are what KSampler/CLIP use.
    model_ref, clip_ref = ["1", 0], ["1", 1]
    for i, (filename, weight, _trigger) in enumerate(resolved):
        node_id = str(100 + i)
        wf[node_id] = {
            "class_type": "LoraLoader",
            "inputs": {
                "lora_name": filename,
                "strength_model": weight,
                "strength_clip": weight,
                "model": model_ref,
                "clip": clip_ref,
            },
        }
        model_ref, clip_ref = [node_id, 0], [node_id, 1]

    wf.update({
        "5":  {"class_type":"LoadImage", "inputs":{"image": image_name}},
        "6":  {"class_type":"VAEEncode", "inputs":{"pixels":["5",0], "vae":["1",2]}},
        "3":  {"class_type":"CLIPTextEncode",
               "inputs":{"text": full_prompt, "clip": clip_ref}},
        "4":  {"class_type":"CLIPTextEncode",
               "inputs":{"text": "", "clip": clip_ref}},
        "7":  {"class_type":"FluxGuidance",
               "inputs":{"conditioning":["3",0], "guidance": 2.5}},
        "8":  {"class_type":"KSampler", "inputs":{
                "model": model_ref,
                "positive":["7",0], "negative":["4",0], "latent_image":["6",0],
                "seed": seed, "steps": 20, "cfg": 1.0,
                "sampler_name": "euler", "scheduler": "simple",
                "denoise": denoise,
              }},
        "11": {"class_type":"VAEDecode", "inputs":{"samples":["8",0], "vae":["1",2]}},
        "9":  {"class_type":"SaveImage", "inputs":{"images":["11",0],
               "filename_prefix": "aquarender"}},
    })
    return wf


def on_generate(_):
    output_box.clear_output()
    btn.disabled = True
    progress_bar.layout.visibility = "visible"
    progress_bar.value = 0
    progress_bar.bar_style = "info"
    try:
        # 1. Read input image
        status_html.value = "<i>Reading input image…</i>"
        img_bytes = _load_input_image()
        progress_bar.value = 5

        # 2. Upload to ComfyUI
        status_html.value = "<i>Uploading to engine…</i>"
        upload = requests.post(
            f"{COMFY_URL}/upload/image",
            files={"image": ("input.png", img_bytes)},
            data={"type": "input", "overwrite": "true"},
            timeout=30,
        ).json()
        image_name = upload["name"]
        progress_bar.value = 15

        # 3. Build workflow (FLUX-dev img2img with stacked LoRAs)
        preset = PRESETS[preset_dd.value]
        seed = seed_box.value or int.from_bytes(os.urandom(4), "big")
        workflow = _build_workflow(
            image_name=image_name,
            prompt=preset["prompt"],
            loras=preset["loras"],
            denoise=min(preset["denoise"] * STRENGTH_MULT[strength_dd.value], 0.95),
            seed=seed,
        )

        # 4. Submit
        status_html.value = "<i>Submitting workflow…</i>"
        queue = requests.post(
            f"{COMFY_URL}/prompt",
            json={"prompt": workflow, "client_id": f"aquarender-{seed}"},
            timeout=30,
        ).json()
        if queue.get("node_errors"):
            progress_bar.bar_style = "danger"
            status_html.value = f"<span style='color:red'>Engine rejected: {queue['node_errors']}</span>"
            return
        prompt_id = queue["prompt_id"]
        progress_bar.value = 25

        # 5. Poll /history
        start = time.time()
        deadline = start + 300
        last_progress = 25
        entry = None
        while time.time() < deadline:
            elapsed = int(time.time() - start)
            status_html.value = f"<i>🎨 Generating… {elapsed}s elapsed</i>"
            try:
                hist = requests.get(f"{COMFY_URL}/history/{prompt_id}", timeout=5).json()
            except Exception:
                hist = {}
            entry = hist.get(prompt_id)
            if entry and entry.get("status", {}).get("completed"):
                if entry["status"].get("status_str") == "error":
                    progress_bar.bar_style = "danger"
                    msgs = entry["status"].get("messages") or []
                    status_html.value = f"<span style='color:red'>Generation error: {msgs}</span>"
                    return
                break
            last_progress = min(last_progress + 2, 90)
            progress_bar.value = last_progress
            time.sleep(0.7)
        else:
            progress_bar.bar_style = "danger"
            status_html.value = "<span style='color:red'>Timed out after 180s</span>"
            return

        progress_bar.value = 92
        status_html.value = "<i>Fetching result…</i>"
        img = entry["outputs"]["9"]["images"][0]
        params = {
            "filename": img["filename"],
            "subfolder": img.get("subfolder", ""),
            "type": img.get("type", "output"),
        }
        r = requests.get(f"{COMFY_URL}/view?{urllib.parse.urlencode(params)}", timeout=30)
        OUTPUT_PATH.write_bytes(r.content)

        progress_bar.value = 100
        progress_bar.bar_style = "success"
        total_s = int(time.time() - start)
        status_html.value = f"<b style='color:green'>✓ Done in {total_s}s — seed {seed}</b>"

        # Inline preview + base64 download button (works on Kaggle and Colab)
        b64 = base64.b64encode(OUTPUT_PATH.read_bytes()).decode()
        download_html = (
            f'<a download="aquarender_seed{seed}.png" href="data:image/png;base64,{b64}" '
            'style="display:inline-block;margin-top:10px;padding:10px 24px;background:#4CAF50;'
            'color:white;text-decoration:none;border-radius:6px;font-weight:bold;font-size:14px;">'
            '📥 Download PNG</a>'
        )
        with output_box:
            display(HTML("<h4>Watercolor:</h4>"))
            display(Image(str(OUTPUT_PATH), width=512))
            display(HTML(download_html))
    except Exception as e:
        progress_bar.bar_style = "danger"
        status_html.value = f"<span style='color:red'>Error: {type(e).__name__}: {e}</span>"
    finally:
        btn.disabled = False


btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML("<h2 style='margin:0 0 10px 0'>🎨 AquaRender</h2>"),
    widgets.HBox([input_mode]),
    url_box,
    file_uploader,
    widgets.HTML("<hr style='margin:10px 0'>"),
    widgets.HBox([preset_dd, strength_dd]),
    seed_box,
    btn,
    progress_bar,
    status_html,
    output_box,
]))
